# **STEP 1: Install Required Libraries**

In [70]:
!pip install -q pymupdf faiss-cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.6 MB/s eta 0:00:00


In [9]:
!pip install -q langchain-text-splitters

# **STEP 2: Import Libraries**

In [71]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# **Step 3: Upload files**

In [11]:
from google.colab import files
uploaded = files.upload()

Saving Statistics-part#1.pdf to Statistics-part#1.pdf
Saving Statistics-part#2.pdf to Statistics-part#2.pdf


# **STEP 4: Load Both PDFs**

In [72]:
import fitz
import re

def load_pdf(file):
    doc = fitz.open(file)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

pdf1 = clean_text(load_pdf("Statistics-part#1.pdf"))
pdf2 = clean_text(load_pdf("Statistics-part#2.pdf"))

all_text = pdf1 + " " + pdf2

# **STEP 5: Split Text into Chunks**

In [74]:
!pip install -q langchain-text-splitters

In [75]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.create_documents([all_text])

# **STEP 6 — VECTOR STORE**

In [77]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

db = FAISS.from_documents(docs, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# **STEP 8 — FINAL SMART AI TUTOR**

In [110]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [115]:
def ask_ai(query):

    # Step 1: retrieve relevant text
    docs = db.similarity_search(query, k=3)

    context = " ".join([d.page_content for d in docs])

    # Step 2: force AI to behave like teacher
    prompt = f"""
You are an expert Statistics teacher.

Rules:
- Do NOT copy text from context
- Ignore spelling mistakes in context
- Always explain in simple English
- Give real-life example
- If possible, suggest statistical test

Format:
Heading:
Simple Explanation:
Real-life Example:
(if needed) Suggested Test:

Question: {query}

Context:
{context}

Answer:
"""

    # Step 3: generate answer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_length=300)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [119]:
print(ask_ai("what is statistics?"))

a science


In [127]:
print(ask_ai("What is the range of co-efficient of correlation?"))

-I to +1


In [132]:
print(ask_ai("what is correlation"))


measure of relationship or interdependence between two variables


In [138]:
print(ask_ai("when to use z-test?"))

psychological and education testing


In [140]:
print(ask_ai("define z-test?"))

The values so obtained are called lhe standard Z scores. Thus a standard Z score is gh by the relation jx-x) Z =SO+ lv-S- 4.6
